Fixed SQL error in traveler bookings query (corrected column name and added ACTIVITY join for price)
*Co-authored with CoCo*

# Lab: CoCo in Snowsight

📚 In this lab you will learn and practice the following:

**CoCo Capabilities in Snowsight**:

❄️ Generate SQL queries from natural language prompts

❄️ Fix SQL errors using AI-assisted error resolution

❄️ Explain complex SQL and Python code

❄️ Use multi-turn conversations to iteratively refine queries

❄️ Explore datasets using natural language questions

❄️ Ground CoCo in your schema using @ references

❄️ Discover built-in skills for domain-specific workflows

❄️ Build a Streamlit in Workspaces app from a natural language description

**Key Concepts**: **CoCo**: A data-native AI-powered coding assistant built into Snowsight
**@ References**: Type `@TABLE_NAME` to ground CoCo in your actual schema
**Skills**: Specialized built-in workflows for domains like governance, cost, security, ML, and dbt
**Multi-Turn Conversation**: CoCo remembers context so you can refine queries iteratively
**Streamlit in Workspaces**: Build and run Streamlit apps directly in your workspace. CoCo generates the code and config files

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).


---

## Introduction to CoCo

CoCo is a truly data-native AI-powered coding assistant available directly in Snowsight.

**How can CoCo help you in the UI?**

| Capability | Description |
|------------|-------------|
| **Write SQL** | Ask in plain English, get executable queries |
| **Fix Errors** | Paste an error or say "fix this" |
| **Explain Code** | Select code and ask what it does |
| **Explore Data** | Ask questions about your tables and schemas |
| **Build Streamlit Apps** | Describe what you want, CoCo generates `streamlit_app.py` and `snowflake.yml` in your workspace |
| **dbt Projects** | Create, run, test, and deploy dbt models |
| **Data Governance** | Masking policies, classification, access controls |
| **Cost Analysis** | Credit usage, warehouse spend, anomalies |
| **Security** | Threat detection, access auditing, network policies |
| **ML Workflows** | Train, register, and deploy models |
| **Notebooks** | Create and edit notebook cells |
| **@ References** | Type `@TABLE_NAME` to ground CoCo in your actual schema |

**Dataset: TravelBug**
- `GENAI_DB.RAW.ACTIVITY` - Travel activities
- `GENAI_DB.RAW.BOOKING` - Bookings with status and payment info
- `GENAI_DB.RAW.TRAVELER` - Travelers
- `GENAI_DB.RAW.REVIEW` - Free-text reviews

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

### Set up your current context for the role, database, schema and warehouse.

Before you begin, make sure your session context is configured correctly. Run the cell below to set your role, database, schema, and warehouse.

In [ ]:
# Setup Context
from snowflake.snowpark.context import get_active_session
import requests
import json
import os

session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'

print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r dataframe_3
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: LLM Functions Part 1';
SHOW PARAMETERS LIKE 'query_tag' in session 
->> SELECT "value" AS query_tag FROM $1;


## Generate SQL from Natural Language

**How to use:** Click on a SQL cell → Open CoCo panel (lightning bolt icon) → Type a natural language prompt.

**Try these prompts in the CoCo chat:**
1. "Show the top 5 most expensive activities with their names and prices"
2. "Which travelers have the most bookings? Show top 10 with their names"
3. "What is the monthly revenue trend from bookings that are confirmed?"

The cell below was generated by CoCo using a natural language prompt.

## Fix Errors with CoCo

**How to use:** Copy and paste the error to the CoCo chat.

**Demo:** Run the broken query below - then ask CoCo to fix it!

In [ ]:
%%sql -r Fix_Errors_with_Cortex_Code_sql
-- Fixed: corrected join column name and joined ACTIVITY for price
SELECT 
    t.TRAVELER_NAME,
    COUNT(b.BOOKING_ID) as total_bookings,
    SUM(a.PRICE) as total_spent
FROM {{user}}_GENAI_DB.RAW.TRAVELER t
JOIN {{user}}_GENAI_DB.RAW.BOOKING b ON t.TRAVELER_ID = b.TRAVELER_ID
JOIN {{user}}_GENAI_DB.RAW.ACTIVITY a ON b.ACTIVITY_ID = a.ACTIVITY_ID
GROUP BY t.TRAVELER_NAME
ORDER BY total_spent DESC
LIMIT 10;


## Explore Your Dataset with CoCo

**How to use:** Ask CoCo questions about your data in plain English. It writes and runs the exploratory queries for you.

**Try asking:**
- "Which activity generates the most revenue?"
- "What's the average time between booking date and activity date?"

**Demo:** Ask the questions above and CoCo will generate the SQL, execute it, and return the results - no manual query writing needed.

## Explain Code with CoCo

**How to use:** Highlight any SQL or Python code, then ask CoCo "Explain this code" or "What does this do?"

**Try asking:**
- "Explain this query step by step"
- "What does this CTE do?"
- "Why is NULLIF used here?"

**Demo:** Select the complex query in the cell below and ask: "Explain this query step by step"

In [ ]:
%%sql -r Explain_Code_with_Cortex_Code_sql
-- Ask CoCo: "Explain this query"
WITH booking_stats AS (
    SELECT 
        a.NAME AS activity_name,
        a.LOCATION,
        COUNT(b.BOOKING_ID) AS total_bookings,
        SUM(CASE WHEN b.BOOKING_STATUS = 'Confirmed' THEN 1 ELSE 0 END) AS confirmed_bookings,
        ROUND(AVG(b.TOTAL_PRICE), 2) AS avg_booking_price,
        ROUND(SUM(b.TOTAL_PRICE), 2) AS total_revenue
    FROM GENAI_DB.RAW.ACTIVITY a
    LEFT JOIN GENAI_DB.RAW.BOOKING b ON a.ACTIVITY_ID = b.ACTIVITY_ID
    GROUP BY a.NAME, a.LOCATION
)
SELECT 
    activity_name,
    location,
    total_bookings,
    confirmed_bookings,
    ROUND(confirmed_bookings * 100.0 / NULLIF(total_bookings, 0), 1) AS confirmation_rate_pct,
    avg_booking_price,
    total_revenue
FROM booking_stats
ORDER BY total_revenue DESC;


## Multi-Turn Conversation and Iterative Refinement

**How to use:** Ask a follow-up question to refine your previous query. CoCo remembers context within the conversation.

**Demo flow:**
1. Ask: "Show me all bookings from January 2026"
2. Then ask: "Add the traveler name and activity name to that query"
3. Then ask: "Only show confirmed bookings and sort by price descending"

Each follow-up builds on the previous result. No need to repeat context!

## Ground CoCo with Schema References

**How to use:** Type `@` in the chat to reference tables, views, or workspace files. This gives CoCo direct access to the object's schema or file contents.

**What you can reference:**
- `@DATABASE.SCHEMA.TABLE` - attaches column names and types
- `@filename.py` - attaches full file content

**Demo flow:**
1. Type: "@GENAI_DB.RAW.BOOKING - what columns does this table have?"
2. Then: "Write a query joining @GENAI_DB.RAW.BOOKING with @GENAI_DB.RAW.TRAVELER"
3. Or attach a file: "Explain @genai_db.resources.genai2day/sis_apps/
travelbug_search_app.py and explain what it does". 

The `@` reference ensures CoCo uses your actual schema - no hallucinated column names!

## Discover Built-in Skills

**How to use:** Ask CoCo "show skills" or "what skills do you have?" to see all available built-in skills organized by category.

**Try asking:**
- "Show skills" - lists all available skill categories
- "What skills do you have for data governance?"
- "Show me security-related skills"

Skills are specialized workflows that give CoCo deep domain expertise in areas like data governance, cost analysis, security, dbt, Streamlit, ML, and more.

## Build a Streamlit App with CoCo

Streamlit is a Python library that lets you build interactive web apps using just Python — no HTML, CSS, or JavaScript needed.

Ask CoCo to build a KPI dashboard for TravelBug. Copy and paste the prompt below into the CoCo chat panel:

```
Build a Streamlit in Workspaces app in this workspace called TRAVELBUG_KPI_DASHBOARD that shows KPIs for TravelBug.

Logo:
- The TravelBug logo is at @{{user}}_GENAI_DB.RESOURCES.GENAI2DAY/sis_apps/travelbug_adventure_awaits.jpg
- Use session.file.get() to download it to a temp directory, then display with st.image()

Data source:
- Table: {{user}}_GENAI_DB.PRESENTATION.TRAVELER_ACTIVITY
- Columns: BOOKING_ID, TRAVELER_ID, TRAVELER_NAME, TOTAL_PRICE, ACTIVITY_NAME, ACTIVITY_LOCATION, BOOKING_STATUS, PAYMENT_STATUS, BOOKING_DATE, ACTIVITY_DATE

KPI metrics (4 columns):
- Total Bookings (count of rows)
- Total Revenue (sum of TOTAL_PRICE)
- Average Booking Price (mean of TOTAL_PRICE)
- Unique Travelers (distinct count of TRAVELER_ID)

Charts (2 columns):
- Bar chart: Revenue by Activity (ACTIVITY_NAME vs sum of TOTAL_PRICE)
- Bar chart: Bookings by Location (ACTIVITY_LOCATION vs count of BOOKING_ID)

Technical requirements:
- Use Snowpark session for data access
- Use pandas for data manipulation
- Use st.bar_chart() for visualizations
- Use st.metric() for KPI display
```

CoCo will create the `streamlit_app.py` and `snowflake.yml` files in this workspace and configure the app for you.

📌 **Note:** After CoCo creates the app, click the **Run** button in the Streamlit file to launch the dashboard.

## 🎯 Challenge Questions

Test your understanding of CoCo capabilities covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What does the @ symbol do when used in the CoCo chat?", "options": ["A) It tags another user in the conversation", "B) It references a table, view, or file to ground CoCo in your actual schema", "C) It runs the referenced SQL query immediately", "D) It switches CoCo to a different language mode"], "hash": "ab8089019ec584f0660fb2062dadf48d"},
    {"q": "What are Skills in CoCo?", "options": ["A) Pre-built keyboard shortcuts for common SQL operations", "B) Specialized built-in workflows that give CoCo deep domain expertise in areas like governance, security, and ML", "C) Automated scripts that run on a scheduled basis", "D) Custom chart types for Snowsight dashboards"], "hash": "b40d1ad1fbf9010f7a986b239e18c5f8"},
    {"q": "How does multi-turn conversation help when using CoCo?", "options": ["A) It allows multiple users to edit the same query simultaneously", "B) CoCo remembers context so you can iteratively refine queries without repeating yourself", "C) It automatically saves every query to a history table", "D) It runs multiple queries in parallel for faster results"], "hash": "c56321480e3a365c1d2035e78eba2136"},
    {"q": "Which command resets the CoCo conversation context?", "options": ["A) /reset", "B) /stop", "C) /clear", "D) /new"], "hash": "21bb3832fb00c031e60206929bec8c89"},
    {"q": "Where can you use CoCo in Snowsight?", "options": ["A) Only in Snowflake Notebooks", "B) Only in SQL Worksheets", "C) In both SQL Worksheets and Notebooks, supporting SQL and Python cells", "D) Only in Streamlit apps deployed to Snowflake"], "hash": "c21e18aebc3a0dcca9c3500e90c65a5c"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '\u2705' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ **Natural Language to SQL**: CoCo translates plain English prompts into executable SQL, eliminating the need to remember exact syntax.

❄️ **AI-Assisted Error Fixing**: Pasting an error or selecting broken code in the CoCo chat instantly surfaces a corrected version with explanation.

❄️ **Schema-Grounded Responses**: Using `@TABLE_NAME` references ensures CoCo works with your actual column names, preventing hallucinated schema.

❄️ **Iterative Refinement via Multi-Turn**: CoCo maintains conversation context so you can build on previous queries step by step without repeating yourself.

❄️ **Built-in Domain Skills**: Specialized skills provide deep expertise in governance, cost analysis, security, ML, dbt, and more, invoked through natural language.

❄️ **Streamlit in Workspaces**: CoCo can scaffold full Streamlit apps directly in your workspace, generating `streamlit_app.py` and `snowflake.yml` from a natural language description, including KPI dashboards and data visualizations.

**Features Demonstrated Today**:

| Feature | How to Access | Benefit |
|---------|---------------|---------|
| Generate SQL | Ask in natural language | Write queries in seconds |
| Fix Errors | Click "Fix" on error or ask "fix this" | No more Googling syntax |
| Explain Code | Select code + "Explain this" | Understand complex logic |
| Multi-Turn | Ask follow-up questions | Iterative refinement |
| Explore Data | Ask questions about your data | Fast exploration |
| @ References | Type `@` + table/file name | Grounded, accurate queries |
| Built-in Skills | "Show skills" or ask domain questions | Deep expertise on demand |
| Streamlit in Workspaces | Describe the app you want | Full app generated in workspace |

📌 **Pro Tips**:
- Use `@` to reference tables/views directly in the chat
- CoCo understands your schema, so ask about columns and relationships
- Works in both SQL worksheets and Notebooks
- Supports Python cells too. Ask it to write pandas/Snowpark code!
- Use `/clear` to reset context when switching topics
- Conversations are ephemeral. Accept code into cells to persist it